# Documentação e tags de PII — `customers`, `sellers` e seus históricos (etapas 2.5 e 3.3)

Aplica **comentários de coluna** e **tags** nas quatro tabelas silver que têm dados cadastrais: `customers`, `sellers` (estado atual, SCD1) e `customers_historico`, `sellers_historico` (uma linha por versão, SCD2; etapa 3.3). É a base que a Fase 6 (governança e LGPD) usa para mascarar colunas e conceder acesso por perfil.

**Tags usadas** (chave = valor)

| Tag | Valores | Onde | Significado |
|---|---|---|---|
| `pii` | `true` | coluna | a coluna é dado pessoal ou permite identificar uma pessoa |
| `pii_categoria` | `identificador`, `localizacao` | coluna | tipo do dado pessoal; a Fase 6 escolhe a máscara por categoria |
| `contem_pii` | `true` / `false` | tabela | a tabela tem alguma coluna com `pii = true` |

**Decisões de classificação**

- `customer_unique_id` é o identificador estável da pessoa: PII, categoria `identificador`.
- Cidade, UF e prefixo de CEP, juntos, localizam uma pessoa: PII, categoria `localizacao`. Em `sellers`, a localização é da empresa vendedora; mesmo assim ela recebe a tag, por prudência: a Fase 6 decide se mascara.
- O **histórico** tem as mesmas colunas PII e a mesma classificação: ele guarda os endereços antigos, então é a tabela mais sensível do cadastro. A exclusão de dados pessoais do histórico é tema da Fase 6.
- `__START_AT` e `__END_AT` são colunas de controle do AUTO CDC (intervalo de validade da versão); não são PII.
- `customer_id` **não** é PII: é um código gerado por pedido, sem relação com a pessoa fora do dataset.

**O notebook é idempotente:** `COMMENT` e `SET TAGS` sobrescrevem o valor anterior, então rodar de novo não muda nada.

**Atenção:** as tabelas são *streaming tables* mantidas pelo pipeline. Depois de rodar este notebook, execute o pipeline de novo (normal) e rode a **última célula**: ela confere se os comentários e as tags continuaram lá.

In [0]:
# O QUE FAZ: define o catálogo e, para cada tabela, a descrição de cada coluna e as tags de PII.
#   Nada é gravado aqui.
#
# RECURSOS:
# - dict {tabela: {coluna: (comentário, tags)}}: a documentação fica em dados, e o código que a
#   aplica (próxima célula) é um só para as duas tabelas.
# - tags: dict {chave: valor}; vazio {} = a coluna não é PII.
CATALOG = "olist"

AUDITORIA = {
    "_ingested_at": ("Momento em que a linha chegou à bronze.", {}),
    "_source_file": ("Arquivo da landing de onde a linha veio.", {}),
    "_source_file_modified_at": ("Data de modificação do arquivo de origem.", {}),
    "_run_id": ("Identificador da execução de ingestão que trouxe a linha.", {}),
}

CDC = {
    "operation": ("Última operação recebida para a chave: INSERT, UPDATE ou DELETE. DELETE não remove a linha da silver; o histórico de versões está em <entidade>_historico.", {}),
    "change_sequence": ("Versão da linha na chave: 1 = INSERT, 2 = UPDATE, 3 = DELETE. Decide qual versão vale.", {}),
    "change_ts": ("Momento da mudança no sistema de origem.", {}),
    "sim_date": ("Data de simulação em que o arquivo foi gerado.", {}),
}

# colunas do histórico: operation, change_sequence, change_ts, sim_date e auditoria existem também lá
CDC_HISTORICO = {
    "__START_AT": ("Início da validade da versão (change_ts em que passou a valer). Coluna do AUTO CDC.", {}),
    "__END_AT": ("Fim da validade da versão (change_ts da próxima versão ou da exclusão). Nulo = versão vigente. Coluna do AUTO CDC.", {}),
    "operation": ("Operação que originou a versão: INSERT ou UPDATE (o DELETE só fecha a versão anterior, não vira linha).", {}),
    "change_sequence": ("Versão da linha na chave: 1 = INSERT, 2 = UPDATE.", {}),
    "change_ts": ("Momento da mudança no sistema de origem; é a coluna que ordena as versões.", {}),
    "sim_date": ("Data de simulação em que o arquivo foi gerado.", {}),
}

PII_LOCALIZACAO = {"pii": "true", "pii_categoria": "localizacao"}
PII_IDENTIFICADOR = {"pii": "true", "pii_categoria": "identificador"}

DOCUMENTACAO = {
    "silver.customers": {
        "tabela": ("Clientes, uma linha por customer_id (última versão do feed de mudanças). Contém dados pessoais.", {"contem_pii": "true"}),
        "colunas": {
            "customer_id": ("Código do cliente no pedido (um por pedido). Chave da tabela. Não é PII.", {}),
            "customer_unique_id": ("Identificador estável da pessoa. Dado pessoal. Nulo quando operation = DELETE.", PII_IDENTIFICADOR),
            "customer_zip_code_prefix": ("Prefixo do CEP (5 dígitos, texto para preservar zeros à esquerda). Dado pessoal.", PII_LOCALIZACAO),
            "customer_city": ("Cidade do cliente. Dado pessoal.", PII_LOCALIZACAO),
            "customer_state": ("UF do cliente. Dado pessoal.", PII_LOCALIZACAO),
            **CDC,
            **AUDITORIA,
        },
    },
    "silver.sellers": {
        "tabela": ("Vendedores, uma linha por seller_id (última versão do feed de mudanças).", {"contem_pii": "true"}),
        "colunas": {
            "seller_id": ("Código do vendedor. Chave da tabela.", {}),
            "seller_zip_code_prefix": ("Prefixo do CEP do vendedor (texto, 5 dígitos). Localização.", PII_LOCALIZACAO),
            "seller_city": ("Cidade do vendedor. Localização.", PII_LOCALIZACAO),
            "seller_state": ("UF do vendedor. Localização.", PII_LOCALIZACAO),
            **CDC,
            **AUDITORIA,
        },
    },
    "silver.customers_historico": {
        "tabela": ("Histórico (SCD2) de clientes: uma linha por versão do cadastro, com __START_AT e __END_AT (nulo = vigente). A exclusão fecha a versão. Contém dados pessoais, inclusive endereços antigos.", {"contem_pii": "true"}),
        "colunas": {
            "customer_id": ("Código do cliente no pedido (um por pedido). Chave da versão. Não é PII.", {}),
            "customer_unique_id": ("Identificador estável da pessoa. Dado pessoal.", PII_IDENTIFICADOR),
            "customer_zip_code_prefix": ("Prefixo do CEP nesta versão (texto, 5 dígitos). Dado pessoal.", PII_LOCALIZACAO),
            "customer_city": ("Cidade do cliente nesta versão. Dado pessoal.", PII_LOCALIZACAO),
            "customer_state": ("UF do cliente nesta versão. Dado pessoal.", PII_LOCALIZACAO),
            **CDC_HISTORICO,
            **AUDITORIA,
        },
    },
    "silver.sellers_historico": {
        "tabela": ("Histórico (SCD2) de vendedores: uma linha por versão do cadastro, com __START_AT e __END_AT (nulo = vigente). A exclusão fecha a versão.", {"contem_pii": "true"}),
        "colunas": {
            "seller_id": ("Código do vendedor. Chave da versão.", {}),
            "seller_zip_code_prefix": ("Prefixo do CEP do vendedor nesta versão (texto, 5 dígitos). Localização.", PII_LOCALIZACAO),
            "seller_city": ("Cidade do vendedor nesta versão. Localização.", PII_LOCALIZACAO),
            "seller_state": ("UF do vendedor nesta versão. Localização.", PII_LOCALIZACAO),
            **CDC_HISTORICO,
            **AUDITORIA,
        },
    },
}


## Aplicar comentários e tags

Para cada coluna, dois comandos SQL do Unity Catalog:

- `ALTER TABLE t ALTER COLUMN c COMMENT '...'`: grava a descrição;
- `ALTER TABLE t ALTER COLUMN c SET TAGS ('chave' = 'valor', ...)`: grava as tags. Coluna sem PII não recebe tag; se tivesse uma de uma execução anterior, ela seria removida com `UNSET TAGS`, assim o resultado final sempre espelha o dicionário.

Se alguma coluna do dicionário não existir na tabela, o comando falha de propósito: é melhor parar do que documentar uma coluna que não está lá.

In [0]:
# O QUE FAZ: percorre o dicionário DOCUMENTACAO e aplica o comentário da tabela, as tags da tabela, o
#   comentário de cada coluna e as tags de PII de cada coluna.
#
# FUNÇÕES DECLARADAS:
# - texto(valor): troca aspas simples por duas, para o texto caber dentro de um literal SQL.
# - tags_sql(tags): monta o trecho ('chave' = 'valor', ...) de um SET TAGS.
#
# FUNÇÕES NATIVAS:
# - spark.sql(comando): executa um comando SQL.
# - ALTER TABLE ... SET TAGS / UNSET TAGS: grava ou remove tags na tabela (Unity Catalog).
# - COMMENT ON TABLE t IS '...': descrição da tabela.
# - ALTER TABLE ... ALTER COLUMN c COMMENT '...' / SET TAGS (...) / UNSET TAGS (...): o mesmo, por coluna.
# - dict.items(): percorre pares (chave, valor).
# - ", ".join(lista): junta os textos com vírgula.
def texto(valor: str) -> str:
    return valor.replace("'", "''")


def tags_sql(tags: dict) -> str:
    return ", ".join(f"'{k}' = '{texto(v)}'" for k, v in tags.items())


TAGS_POSSIVEIS = ("pii", "pii_categoria")

for nome, doc in DOCUMENTACAO.items():
    tabela = f"{CATALOG}.{nome}"
    comentario, tags = doc["tabela"]
    spark.sql(f"COMMENT ON TABLE {tabela} IS '{texto(comentario)}'")
    spark.sql(f"ALTER TABLE {tabela} SET TAGS ({tags_sql(tags)})")

    for coluna, (comentario, tags) in doc["colunas"].items():
        spark.sql(f"ALTER TABLE {tabela} ALTER COLUMN {coluna} COMMENT '{texto(comentario)}'")
        if tags:
            spark.sql(f"ALTER TABLE {tabela} ALTER COLUMN {coluna} SET TAGS ({tags_sql(tags)})")
        else:
            lista = ", ".join(f"'{t}'" for t in TAGS_POSSIVEIS)
            spark.sql(f"ALTER TABLE {tabela} ALTER COLUMN {coluna} UNSET TAGS ({lista})")
    print(f"{tabela}: {len(doc['colunas'])} colunas documentadas")

## Conferir

Lê o que ficou gravado, direto do catálogo:

- `information_schema.column_tags`: uma linha por tag de coluna;
- `describe table`: confere se há coluna sem comentário.

**Rode esta célula duas vezes:** agora e depois de executar o pipeline de novo. Se na segunda vez as tags ou comentários sumirem, o pipeline os recria e a descrição das colunas precisa passar para dentro do pipeline.
**Etapa 3.3:** depois de rodar, execute o pipeline com *Full refresh* só se quiser provar que o histórico também sobrevive (as tabelas de histórico são recriadas), e rode esta célula de novo.


In [0]:
# O QUE FAZ: mostra as tags de coluna gravadas e lista colunas sem comentário (deve vir vazio).
#
# FUNÇÕES NATIVAS:
# - spark.table("olist.information_schema.column_tags"): visão do catálogo com as tags de coluna
#   (colunas: schema_name, table_name, column_name, tag_name, tag_value).
# - spark.sql("DESCRIBE TABLE t"): devolve col_name, data_type e comment de cada coluna.
# - coluna.isin(lista): verdadeiro se o valor está na lista; a lista vem das tabelas do dicionário.
# - DataFrame.where / orderBy / select: filtrar, ordenar, escolher colunas.
# - display(df): mostra o DataFrame como tabela interativa.
from pyspark.sql import functions as F

display(
    spark.table(f"{CATALOG}.information_schema.column_tags")
    .where((F.col("schema_name") == "silver") & F.col("table_name").isin([n.split(".")[1] for n in DOCUMENTACAO]))
    .select("table_name", "column_name", "tag_name", "tag_value")
    .orderBy("table_name", "column_name", "tag_name")
)

for nome in DOCUMENTACAO:
    sem_comentario = (
        spark.sql(f"DESCRIBE TABLE {CATALOG}.{nome}")
        .where(F.col("col_name").isNotNull() & ~F.col("col_name").startswith("#") & (F.col("col_name") != ""))
        .where(F.col("comment").isNull() | (F.col("comment") == ""))
        .select("col_name")
    )
    print(f"{nome}: colunas sem comentário -> {[r.col_name for r in sem_comentario.collect()]}")